# Clase 157 — Variational Autoencoders (VAE)

El **VAE** (Kingma & Welling, 2014) es la versión probabilística del AE: el
encoder devuelve `(μ, log σ²)` de una gaussiana, se muestrea `z` con el
**reparametrization trick** `z = μ + σ·ε`, y la loss es el **ELBO**
= `reconstruction + β·KL(q(z|x) ‖ N(0,I))`. Resultado: latent continuo → permite
generar e interpolar.

**Requiere:** `tensorflow` / `keras`. El código es la **API real de Keras**
(capa custom + `train_step`); no se ejecuta sin TF. La fórmula de KL se ilustra
además en numpy.

## 🧠 Intuición previa

**VAE = comprimir a un latente del que se puede *muestrear*.** Un autoencoder normal comprime cada imagen a un **punto** del espacio latente; si muestreás un punto al azar, sueles caer en un "hueco" y el decoder produce basura. El VAE, en cambio, obliga al encoder a devolver una **distribución** `(μ, σ)` por muestra y empuja el conjunto a parecerse a una gaussiana estándar `N(0, I)` (término KL). Así el latente queda **continuo y sin huecos**: muestrear `z ~ N(0, I)` y pasarlo por el decoder **genera** ejemplos nuevos y verosímiles, e interpolar entre dos `z` da transiciones suaves.

## 1. Entorno

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TF = True
    tf.random.set_seed(42)
    print('tensorflow:', tf.__version__)
except Exception as e:
    HAS_TF = False
    print('tensorflow no instalado. Motivo:', type(e).__name__)

import numpy as np
np.random.seed(42)
latent_dim = 2

## 2. Capa de muestreo (reparametrization trick)

`z = μ + exp(0.5·log σ²)·ε`, con `ε ~ N(0, I)`. El ruido `ε` es externo, así los
gradientes fluyen por `μ` y `σ`.

In [ ]:
if HAS_TF:
    class Sampling(layers.Layer):
        def call(self, inputs):
            z_mean, z_log_var = inputs
            eps = tf.random.normal(shape=tf.shape(z_mean))
            return z_mean + tf.exp(0.5 * z_log_var) * eps
    print('Capa Sampling definida (reparametrization trick).')
else:
    # ilustracion numpy del truco
    z_mean, z_log_var = np.zeros(4), np.zeros(4)
    eps = np.random.normal(size=4)
    z = z_mean + np.exp(0.5 * z_log_var) * eps
    print('z (numpy) =', np.round(z, 3))

## 3. Encoder: `x → (z_mean, z_log_var) → z`

Con la Functional API se devuelven las tres salidas.

In [ ]:
if HAS_TF:
    enc_in = keras.Input(shape=(784,))
    h = layers.Dense(256, activation='relu')(enc_in)
    z_mean = layers.Dense(latent_dim, name='z_mean')(h)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(h)
    z = Sampling()([z_mean, z_log_var])
    encoder = keras.Model(enc_in, [z_mean, z_log_var, z], name='encoder')
    encoder.summary()
else:
    print('Encoder: 784->256->(z_mean, z_log_var)->Sampling->z')

## 4. Decoder: `z → x_reconstruido`

In [ ]:
if HAS_TF:
    dec_in = keras.Input(shape=(latent_dim,))
    d = layers.Dense(256, activation='relu')(dec_in)
    dec_out = layers.Dense(784, activation='sigmoid')(d)
    decoder = keras.Model(dec_in, dec_out, name='decoder')
    decoder.summary()
else:
    print('Decoder: z(latent_dim)->256->784(sigmoid)')

## 5. Modelo VAE con `train_step` custom (ELBO)

`KL(N(μ,σ²) ‖ N(0,I)) = -0.5·Σ(1 + log σ² - μ² - σ²)`. La loss total es
`reconstruction + KL` y se optimiza con un `GradientTape`.

In [ ]:
if HAS_TF:
    class VAE(keras.Model):
        def __init__(self, encoder, decoder, **kw):
            super().__init__(**kw)
            self.encoder, self.decoder = encoder, decoder

        def train_step(self, data):
            with tf.GradientTape() as tape:
                z_mean, z_log_var, z = self.encoder(data)
                recon = self.decoder(z)
                rec_loss = tf.reduce_sum(
                    keras.losses.binary_crossentropy(
                        data[..., None], recon[..., None]), axis=-1)
                kl = -0.5 * tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
                loss = tf.reduce_mean(rec_loss + kl)
            grads = tape.gradient(loss, self.trainable_weights)
            self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
            return {'loss': loss, 'kl': tf.reduce_mean(kl)}

    vae = VAE(encoder, decoder)
    vae.compile(optimizer='adam')
    print('VAE compilado. Loss = BCE_reconstruccion + KL (ELBO).')
else:
    mu, logv = np.array([0.5, -0.3]), np.array([0.1, -0.2])
    kl = -0.5 * np.sum(1 + logv - mu**2 - np.exp(logv))
    print('KL(numpy) =', round(float(kl), 4))

## 6. Generación: muestrear `z ~ N(0,I)` → decoder

Para generar imágenes nuevas se muestrea del **prior** (no del posterior).

In [ ]:
z_samples = np.random.normal(size=(64, latent_dim)).astype('float32')
if HAS_TF:
    generated = decoder.predict(z_samples, verbose=0)
    print('generadas:', generated.shape)   # (64, 784)
else:
    print('decoder.predict(z ~ N(0,I))  -> 64 imagenes nuevas de 784 pixeles')

# Interpolacion lineal en el latente entre dos puntos z_A, z_B
z_A, z_B = z_samples[0], z_samples[1]
alphas = np.linspace(0, 1, 10)[:, None]
z_interp = (1 - alphas) * z_A + alphas * z_B
print('interpolacion:', z_interp.shape)   # (10, latent_dim) -> transiciones suaves

## Ejercicios

1. **VAE básico**: entrená el VAE en MNIST y verificá que la `loss` baja.
2. **Sampling**: muestreá `z ~ N(0,I)` de tamaño `(100, latent_dim)` y visualizá el grid.
3. **Interpolación**: interpolá entre `z_A` y `z_B` (10 pasos) y mostrá transiciones suaves.
4. **β-VAE**: multiplicá el término KL por `β ∈ {1, 5, 10}` y compará disentanglement vs blur.
5. **Posterior collapse**: con LR alto, verificá `z_mean.std() → 0` (el encoder colapsa).

## Conclusiones

- El VAE aprende una **distribución** `(μ, σ)` sobre el latente, no un punto.
- El reparametrization trick `z = μ + σ·ε` hace diferenciable el muestreo.
- La loss ELBO = `reconstruction + β·KL` equilibra fidelidad y estructura latente.
- Un latente continuo permite **generar** (muestrear del prior) e **interpolar**.
- Los outputs salen borrosos (MSE/BCE) → GANs y difusión mejoran la nitidez.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README de esta clase. El código que usa librerías pesadas (`transformers` / `torch` / `keras` / `diffusers`) es la **API real** de la industria y se valida por sintaxis (los modelos requieren GPU/descarga). Los **núcleos numéricos** están en numpy puro, son **ejecutables** y se autoverifican con `assert`.

### Ejercicio 1 — VAE básico: entrenar y ver bajar la loss (API Keras)

In [ ]:
if HAS_TF:
    from tensorflow import keras
    (Xtr, _), _ = keras.datasets.mnist.load_data()
    Xtr = (Xtr.astype('float32') / 255.0).reshape(-1, 784)
    vae.fit(Xtr, epochs=10, batch_size=128)      # loss = reconstruction + KL (ELBO)
    print('VAE entrenado; la loss total (ELBO) debe bajar epoca a epoca.')
else:
    print('vae.fit(X, epochs=10): la loss = BCE_reconstruccion + KL desciende.')

### Ejercicio 2 — Sampling del prior `N(0,I)` → 100 imágenes

In [ ]:
import numpy as np
z = np.random.normal(size=(100, latent_dim)).astype('float32')
if HAS_TF:
    imgs = decoder.predict(z, verbose=0)         # (100, 784) imagenes nuevas
    print('generadas:', imgs.shape)
else:
    print('decoder.predict(z ~ N(0,I)) -> 100 imagenes generadas (grid 10x10).')
assert z.shape == (100, latent_dim)

### Ejercicio 3 — Interpolación lineal entre `z_A` y `z_B` (ejecutable)

In [ ]:
import numpy as np
# Reutiliza z_A, z_B definidos arriba; 10 pasos suaves en el latente.
alphas = np.linspace(0, 1, 10)[:, None]
z_path = (1 - alphas) * z_A + alphas * z_B
assert z_path.shape == (10, latent_dim)
assert np.allclose(z_path[0], z_A) and np.allclose(z_path[-1], z_B)   # extremos exactos
# monotonia: la distancia a z_A crece a lo largo del camino
d = np.linalg.norm(z_path - z_A, axis=1)
assert np.all(np.diff(d) >= -1e-6)
print('Interpolacion latente OK: extremos = z_A/z_B, transicion monotona.')
if HAS_TF:
    frames = decoder.predict(z_path, verbose=0)  # morphing suave A -> B
    print('frames de interpolacion:', frames.shape)

### Ejercicio 4 — β-VAE: efecto de β sobre el término KL (ejecutable)

In [ ]:
import numpy as np
# KL(N(mu,sigma^2) || N(0,I)) = -0.5 * sum(1 + log s^2 - mu^2 - s^2)
def kl_std_normal(mu, logvar):
    return -0.5 * np.sum(1 + logvar - mu**2 - np.exp(logvar))
assert abs(kl_std_normal(np.zeros(4), np.zeros(4))) < 1e-9   # q == prior -> KL = 0
assert kl_std_normal(np.ones(4), np.zeros(4)) > 0            # posterior alejado -> KL > 0
mu, logv = np.array([1.0, -0.5]), np.array([0.2, -0.3])
for beta in (1, 5, 10):
    print(f'β={beta:2d} -> término β·KL = {beta * kl_std_normal(mu, logv):.3f}')
print('β alto presiona mas hacia N(0,I): mas disentanglement, mas blur.')

### Ejercicio 5 — Posterior collapse: `z_mean.std() → 0`

In [ ]:
if HAS_TF:
    z_mean, z_log_var, _ = encoder.predict(Xtr[:2000], verbose=0)
    s = float(z_mean.std())
    print('z_mean.std() =', round(s, 4))
    print('Si s -> 0, el encoder ignora la entrada (posterior collapse).')
else:
    print('Diagnostico: monitorear z_mean.std(); si colapsa a ~0 el KL domino'
          ' la loss. Mitigar con KL annealing o free bits.')